### data Processing & split 

* After an EDA we realise that we don't need any data processing for this training set, maybe we will redo this decision after analyzing the training metrics.

##### Train / Val / Test Split 

In [1]:
import os
import shutil
import random
from pathlib import Path

In [ ]:
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

images_path = Path('../data/raw/images')
labels_path = Path('../data/raw/labels')

Splits = {
    'train': TRAIN_RATIO,
    'val': VAL_RATIO,
    'test': TEST_RATIO
}

Base = Path('../data/splits')

for split in Splits:
    (Base / split / 'images').mkdir(parents=True, exist_ok=True)
    (Base / split / 'labels').mkdir(parents=True, exist_ok=True)

images_exts = {'.jpg', '.jpeg','.png'}
all_images = sorted([
    f for f in images_path.iterdir()
    if f.suffix.lower() in images_exts
])

paired = [f for f in all_images if (labels_path / (f.stem + '.txt')).exists()]
print(f'Paired image-label files: {len(paired)}')

# shuffle deterministically 
random.seed(SEED)
random.shuffle(paired)

# Compute split boundaries
n       = len(paired)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
# test gets the remainder so total always == n
n_test  = n - n_train - n_val

splits_data = {
    'train': paired[:n_train],
    'val':   paired[n_train:n_train + n_val],
    'test':  paired[n_train + n_val:],
}

# copy files 
for split, files in splits_data.items():
    for img_path in files:
        lbl_path = labels_path / (img_path.stem + '.txt')

        shutil.copy(img_path, Base / split / 'images' / img_path.name)
        shutil.copy(lbl_path, Base / split / 'labels' / lbl_path.name)

    print(f'{split:6s} - {len(files)} images ({len(files)/n*100:.1f}%)')

print('\nDone. Folder structure:')
for split in Splits:
    imgs = len(list((Base / split / 'images').iterdir()))
    lbls = len(list((Base / split / 'labels').iterdir()))
    print(f'  data/{split}/images/ - {imgs} files')
    print(f'  data/{split}/labels/ - {lbls} files')


..\data\raw\images\GX010023_frame_00000_jpg.rf.Q2WZf7zPwjKpg3m7NQ6o.jpg
Paired image-label files: 753
train  - 527 images (70.0%)
val    - 150 images (19.9%)
test   - 76 images (10.1%)

Done. Folder structure:
  data/train/images/ - 527 files
  data/train/labels/ - 527 files
  data/val/images/ - 150 files
  data/val/labels/ - 150 files
  data/test/images/ - 76 files
  data/test/labels/ - 76 files
